# PA3 · Task 1 · Redo of the OPLoRA Proposition-2 check (FP64 reference)

**Why this notebook exists.** The diagnostics in `t1_eval` reported one FAIL: "top-128 singular triples preserved, (W₀ + ΔW) vᵢ = σᵢ uᵢ" in layers 0 and 13 (1.17 × 10⁻⁴ and 1.08 × 10⁻⁴ against a 10⁻⁴ threshold). That check compared the trained matrix with a yardstick, a fresh FP32 SVD of W₀ computed on the GPU, and the yardstick itself is only accurate to about 10⁻⁴. The error it reported is the yardstick's error plus the update's effect.

**What it does.** Redoes the check with an exact yardstick (an FP64 SVD, accurate to about 10⁻¹⁵), with the same 10⁻⁴ threshold, on all 196 matrices OPLoRA adapted (the three diagnostic matrices included). On the GPU it also replays the original check and splits it into its two parts. It trains and evaluates nothing and changes none of the reported diagnostics; its results go to their own folder, `t1_recheck/`.

**Settings:** Accelerator **GPU T4** (×1 is enough; the GPU only replays the original check), Internet **On** (downloads Qwen3-0.6B). About 10–20 min.
**Attach:** `pa3-task1` (code version 2026.10.31 or newer) and the **t1_train output in which OPLoRA finished** (the same one you attached to `t1_eval`).

**Run with "Save Version → Save & Run All".**

**How to read the output:** every check prints `PASS / FAIL / OBSERVE`, then the **expected** and **actual** lines.

In [1]:
# S1. Find the Task 1 code (attached Kaggle dataset, or a clone of your GitHub repo), copy it to /tmp, check the GPUs.
# All GPU work runs in separate processes, so this notebook itself never holds GPU memory.
import os, sys, glob, json, time, shutil, subprocess
from pathlib import Path
SESSION_START = time.time()     # Kaggle's 12-hour limit counts from about now

REPO_URL = ""        # optional: your public GitHub repo URL, used only if no code dataset is attached
REQUIRED_CODE_VERSION = "2026.10.31" # this notebook needs this code version or newer
CODE = Path("/tmp/pa3_task1_code")

def code_version(root):
    import re
    try:
        m = re.search(r'CODE_VERSION\s*=\s*"([^"]+)"', (Path(root) / "common" / "__init__.py").read_text())
        return m.group(1) if m else "0 (no version stamp: older than 2026.10.30)"
    except FileNotFoundError:
        return "0 (no version stamp: older than 2026.10.30)"

# every attached copy of the code, newest version first (a dataset can hold several copies after re-uploads)
hits = sorted({Path(p).parents[1] for p in glob.glob("/kaggle/input/**/task1/t1_train.py", recursive=True)},
              key=code_version, reverse=True)
for h in hits:
    print(f"found code copy: {h}  (version {code_version(h)})")
if CODE.exists():
    shutil.rmtree(CODE)
if hits:
    if code_version(hits[0]) < REQUIRED_CODE_VERSION:
        raise SystemExit(f"The newest attached code is version {code_version(hits[0])}, but this notebook needs "
                         f"{REQUIRED_CODE_VERSION}. Upload the latest pa3-task1.zip as a new dataset version and re-attach it.")
    shutil.copytree(hits[0], CODE)
    print("using:", hits[0])
elif REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(CODE)], check=True)
    print("code cloned from", REPO_URL)
else:
    raise SystemExit("Attach the pa3-task1 dataset (or set REPO_URL) and re-run this cell.")
sys.path.insert(0, str(CODE))
os.chdir(CODE)
from common.utils import Checks, human_bytes, dir_size_bytes, read_json, read_jsonl
CHK = Checks()
CHK.check(code_version(CODE) >= REQUIRED_CODE_VERSION, "code version in use", f">= {REQUIRED_CODE_VERSION}", code_version(CODE))
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
    GPUS = [l.strip() for l in smi.stdout.strip().splitlines() if l.strip()]
except FileNotFoundError:
    GPUS = []
CHK.check(len(GPUS) >= 1, "GPU visible", "GPU T4 x2 (Settings -> Accelerator)", GPUS or "no GPU: is the accelerator switched on?")
import importlib.metadata as md
vers = {p: md.version(p) for p in ["torch", "transformers", "datasets"]}
CHK.observe("library versions", vers)
tv = tuple(int(x) for x in vers["transformers"].split(".")[:2])
if tv < (4, 51):     # Qwen3 support arrived in transformers 4.51
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers"], check=True)
    print("upgraded transformers: restart the kernel and run again")

found code copy: /kaggle/input/datasets/zylah628/pa3-task1/pa3-task1  (version 2026.10.31)
using: /kaggle/input/datasets/zylah628/pa3-task1/pa3-task1
[PASS   ] code version in use
          expected: >= 2026.10.31
          actual  : 2026.10.31
[PASS   ] GPU visible
          expected: GPU T4 x2 (Settings -> Accelerator)
          actual  : ['Tesla T4, 15360 MiB', 'Tesla T4, 15360 MiB']
[OBSERVE] library versions
          actual  : {'torch': '2.11.0+cu128', 'transformers': '5.16.1', 'datasets': '4.8.5'}


In [2]:
# S2. Find the finished OPLoRA run (the newest finished copy among the attached outputs) and confirm it is the reported run.
from task1.t1_eval import find_run
WORK = Path("/kaggle/working"); OUT = WORK / "t1_recheck"
RUN = find_run("oplora")
CHK.check(RUN is not None and (RUN / "adapter_final_projected.pt").exists(), "finished OPLoRA run attached, with its projected factors",
          "t1_runs/oplora/adapter_final_projected.pt", RUN)
if RUN is not None:
    inj, fs = read_json(RUN / "injection.json"), read_json(RUN / "final_summary.json")
    CHK.check("re-orthonormalisation" in (inj.get("basis_method") or ""), "this is the reported OPLoRA run (re-orthonormalised bases)",
              "re-orthonormalised", inj.get("basis_method"))
    CHK.observe("its training-time constraint check on all 196 matrices: ‖Uₖᵀ ΔW‖ / ‖ΔW‖ and ‖ΔW Vₖ‖ / ‖ΔW‖",
                (f"{fs['oplora_worst_rel_UkT_dW']:.2e}", f"{fs['oplora_worst_rel_dW_Vk']:.2e}"), note="the reported run: 1.01e-07 and 1.19e-07")
ORIG = CODE / "results" / "task1" / "t1_diag" / "diag_checks.json"   # the original diagnostics checks, shipped with the code
CHK.observe("original diagnostics checks (for side-by-side comparison)", "found" if ORIG.exists() else "not found", note=str(ORIG))

[PASS   ] finished OPLoRA run attached, with its projected factors
          expected: t1_runs/oplora/adapter_final_projected.pt
          actual  : /kaggle/input/notebooks/zylah628/task1-oplora-run2/t1_runs/oplora
[PASS   ] this is the reported OPLoRA run (re-orthonormalised bases)
          expected: re-orthonormalised
          actual  : exact FP32 SVD, then a one-time FP64 QR re-orthonormalisation of Uk and Vk (column signs kept), stored in FP32
[OBSERVE] its training-time constraint check on all 196 matrices: ‖Uₖᵀ ΔW‖ / ‖ΔW‖ and ‖ΔW Vₖ‖ / ‖ΔW‖
          expected: the reported run: 1.01e-07 and 1.19e-07
          actual  : ('1.01e-07', '1.19e-07')
[OBSERVE] original diagnostics checks (for side-by-side comparison)
          expected: /tmp/pa3_task1_code/results/task1/t1_diag/diag_checks.json
          actual  : found


In [3]:
# S3. The recheck, in a separate process. The FP64 work runs on the CPU; the GPU only replays the original FP32 check.
t0 = time.time()
r = subprocess.run([sys.executable, "-u", "-m", "task1.t1_recheck", "--out", str(OUT), "--original_checks", str(ORIG)],
                   cwd=CODE, capture_output=True, text=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
print(r.stdout[-15000:]); print(r.stderr[-3000:] if r.returncode else "")
RECHECK_OK = r.returncode == 0 and "0 FAIL" in r.stdout
CHK.check(RECHECK_OK, "recheck finished and all its checks passed", "0 FAIL",
          [l for l in r.stdout.splitlines() if l.startswith("====")][-1:] or r.returncode)
CHK.observe("recheck time", f"{(time.time() - t0) / 60:.1f} min")

[PASS   ] finished OPLoRA run with its exported projected factors
          expected: t1_runs/oplora/adapter_final_projected.pt
          actual  : /kaggle/input/notebooks/zylah628/task1-oplora-run2/t1_runs/oplora
[OBSERVE] OPLoRA run being checked
          expected: α / r = 2, k = 128; bases: exact FP32 SVD, then a one-time FP64 QR re-orthonormalisation of Uk and Vk (column signs kept), stored in FP32
          actual  : /kaggle/input/notebooks/zylah628/task1-oplora-run2/t1_runs/oplora
   28 / 196 matrices  (21 s)
   56 / 196 matrices  (41 s)
   84 / 196 matrices  (62 s)
  112 / 196 matrices  (82 s)
  140 / 196 matrices  (103 s)
  168 / 196 matrices  (123 s)
  196 / 196 matrices  (144 s)
[PASS   ] the FP64 reference is exact: W₀Vₖ = UₖΣₖ on all 196 matrices
          expected: < 1e-10
          actual  : 2.5e-15
[PASS   ] redo of the diagnostics check, layer 0 down_proj: top-128 singular triples preserved (FP64 reference)
          expected: (W₀ + ΔW) vᵢ = σᵢ uᵢ and uᵢᵀ (W₀ + ΔW) = σ

In [4]:
# S4. Results for the GitHub repo: t1_recheck.zip (a few kB). Unzip it into results/task1/ (it creates results/task1/t1_recheck/).
import zipfile
files = sorted(OUT.glob("*")) if OUT.exists() else []
with zipfile.ZipFile(WORK / "t1_recheck.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, f"t1_recheck/{f.name}")
CHK.observe("t1_recheck.zip contents", [f.name for f in files], note="download it and unzip it into results/task1/")
CHK.summary(); CHK.save(WORK / "t1_recheck_notebook_checks.json")
print("\nNEXT:", "send this notebook (S2-S4 output) and t1_recheck.zip." if RECHECK_OK else
      "a check failed: send this notebook's output as it is, so we can see which matrix and by how much.")

[OBSERVE] t1_recheck.zip contents
          expected: download it and unzip it into results/task1/
          actual  : ['oplora_prop2_recheck.csv', 'original_check_replay.csv', 'recheck_checks.json']

==== 5 PASS | 0 FAIL | 5 OBSERVE ====

NEXT: send this notebook (S2-S4 output) and t1_recheck.zip.
